In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_nvidia import NVIDIAEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field



In [ ]:
# Document Load
loader = PyPDFLoader("../data/sample_pdf.pdf")
docs = loader.load()


# Split in Chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
docs = splitter.split_documents(docs)

# embeddings and Vector DB
embed = NVIDIAEmbeddings(model="nvidia/nemotron-3-embed-1b")
vector_store = InMemoryVectorStore.from_documents(
    documents=docs,
    embedding=embed
)

In [ ]:
class RagState(BaseModel):
    question: str = Field(description="User Question",default="")
    documents: list = []
    context: str = Field(description="Context data for user questions",default="")
    answer : str = Field(description="Final Answer...",default="" )

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b")

In [ ]:
# Ques -> retrieve -> Generate -> End

def retrieve_node(state:RagState) -> RagState:
    docs = vector_store.similarity_search(query=state.question)
    state.documents = docs
    return state

def create_context(state:RagState) -> RagState:
    context = ""
    for doc in state.documents:
        context += doc.page_content + "\n\n"
    state.context = context
    return state

def generate_node(state:RagState) -> RagState:
    prompt = f"""
        You are the assistant and provide the answer for user question based on the provieded context.
        If you don't find the relevent answer ,then just say 'I dont know.'.
        Context : [{state.context}]
        Question : [{state.question}]
    """
    res = llm.invoke(prompt)
    state.answer = res.content
    return state

In [ ]:
graph = StateGraph(RagState)

graph.add_node('retrieve_node',retrieve_node)                       
graph.add_node('create_context',create_context)                       
graph.add_node('generate_node',generate_node) 

graph.add_edge(START,'retrieve_node')
graph.add_edge('retrieve_node','create_context')
graph.add_edge('create_context','generate_node')
graph.add_edge('generate_node',END)

graph= graph.compile()

In [ ]:
from IPython.display import Image
Image(graph.get_graph().draw_mermaid_png())

In [ ]:
res = graph.invoke({'question':'Tell me about Rubrics'})

In [ ]:
print(res['answer'])